In [ ]:
import os
import shutil
from google.colab import drive


drive.mount('/content/drive')


DRIVE_IMG = "/content/drive/MyDrive/satellitle dataset/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/images"
DRIVE_LBL = "/content/drive/MyDrive/satellitle dataset/extracting_slums_from_satellite_imagery/extracting_slums_from_satellite_imagery/labels"

# 2. LOCAL SYNC (The "Pro" speed fix)
!mkdir -p /content/local_data/images /content/local_data/labels
print("Syncing data to local SSD...")
!cp -r "{DRIVE_IMG}/." /content/local_data/images/
!cp -r "{DRIVE_LBL}/." /content/local_data/labels/
print("Sync Complete.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Syncing data to local SSD...
Sync Complete.


In [ ]:
import glob
import os
from sklearn.model_selection import train_test_split

# Get all image and label paths
all_img_paths = sorted(glob.glob("/content/local_data/images/*.tif"))
all_lbl_paths = sorted(glob.glob("/content/local_data/labels/*.tif"))

# Extract base filenames (without path and extension)
img_basenames = {os.path.basename(p).split('.')[0] for p in all_img_paths}
lbl_basenames = {os.path.basename(p).split('.')[0] for p in all_lbl_paths}

# Find common basenames
common_basenames = sorted(list(img_basenames.intersection(lbl_basenames)))

# Filter paths to include only those with common basenames
all_img_files = []
all_lbl_files = []

for basename in common_basenames:
    img_path = f"/content/local_data/images/{basename}.tif"
    lbl_path = f"/content/local_data/labels/{basename}.tif"

    # Ensure both files actually exist before adding them
    if os.path.exists(img_path) and os.path.exists(lbl_path):
        all_img_files.append(img_path)
        all_lbl_files.append(lbl_path)


# Split: 80% Train, 20% for Val/Test
train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    all_img_files, all_lbl_files, test_size=0.2, random_state=42
)

# Split the 20% into 10% Val and 10% Test
val_imgs, test_imgs, val_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.5, random_state=42
)

In [ ]:
print(f"Total images: {len(all_img_files)}")
print(f"Training images: {len(train_imgs)}")
print(f"Validation images: {len(val_imgs)}")
print(f"Test images: {len(test_imgs)}")

Total images: 2323
Training images: 1858
Validation images: 232
Test images: 233


In [ ]:
import rasterio
import torch
from torch.utils.data import Dataset
import albumentations as A

class SlumDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, i):
        # Read Image (3 bands)
        with rasterio.open(self.images[i]) as src:
            image = src.read([1, 2, 3]).transpose(1, 2, 0) # H,W,C
            image = (image / image.max() * 255).astype('uint8') # Normalize to 8-bit

        # Read Mask
        with rasterio.open(self.labels[i]) as src:
            mask = src.read(1)

        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image, mask = augmented['image'], augmented['mask']

        # To PyTorch Tensor (CxHxW)
        image = torch.from_numpy(image).permute(2, 0, 1).float() / 255.0
        mask = torch.from_numpy(mask).long()
        return image, mask

In [ ]:
!pip install segmentation_models_pytorch
import segmentation_models_pytorch as smp

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 6.2 MB/s eta 0:00:00


In [ ]:
# Augmentations used in the Ardhitasari (2023) / Mumbai research
import albumentations as A
import torch

train_transform = A.Compose([
    A.Resize(256, 256), # Resize to a dimension divisible by 32
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.0625, scale_limit=0.1, rotate_limit=45, p=0.5),
    A.RandomBrightnessContrast(p=0.2),
])

val_test_transform = A.Compose([
    A.Resize(256, 256),
])

# BATCH SIZE
BATCH_SIZE = 8

# Create Datasets
train_dataset = SlumDataset(train_imgs, train_lbls, transform=train_transform)
val_dataset = SlumDataset(val_imgs, val_lbls, transform=val_test_transform)
test_dataset = SlumDataset(test_imgs, test_lbls, transform=val_test_transform)

# Create DataLoaders
train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE, shuffle=True, num_workers=0
)

val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

test_loader = torch.utils.data.DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE, shuffle=False, num_workers=0
)

/usr/local/lib/python3.12/dist-packages/albumentations/core/validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [ ]:
import torch.nn as nn

# 1. Model Selection (VGG16-FPN)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = smp.FPN(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation='sigmoid'
).to(device)

# 2. Hybrid Loss Function (Binary Cross Entropy + Dice)
class ArdhitasariHybridLoss(nn.Module):
    def __init__(self):
        super().__init__()
        self.bce = nn.BCELoss()
        self.dice = smp.losses.DiceLoss(mode='binary')

    def forward(self, pred, target):
        # Target must be float for BCE and shape must match (B, 1, H, W)
        return self.bce(pred, target.float().unsqueeze(1)) + self.dice(pred, target)

criterion = ArdhitasariHybridLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

In [ ]:
from tqdm import tqdm

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

epochs = 30
best_f1 = 0.0
accumulation_steps = 4

for epoch in range(epochs):

    model.train()
    train_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")

    optimizer.zero_grad()

    for i, (imgs, msks) in enumerate(pbar):
        imgs, msks = imgs.to(device), msks.to(device)

        preds = model(imgs)
        loss = criterion(preds, msks)


        loss = loss / accumulation_steps
        loss.backward()

        if (i + 1) % accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        train_loss += (loss.item() * accumulation_steps)
        pbar.set_postfix(loss=f"{(loss.item() * accumulation_steps):.4f}", lr=f"{optimizer.param_groups[0]['lr']:.6f}")


    model.eval()
    val_stats = []

    with torch.no_grad():
        for v_imgs, v_msks in tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]"):
            v_imgs, v_msks = v_imgs.to(device), v_msks.to(device)
            v_preds = model(v_imgs)

            # Record TP, FP, FN, TN
            stats = smp.metrics.get_stats(v_preds, v_msks.unsqueeze(1), mode='binary', threshold=0.5)
            val_stats.append(stats)

    tp = torch.cat([x[0] for x in val_stats]).sum()
    fp = torch.cat([x[1] for x in val_stats]).sum()
    fn = torch.cat([x[2] for x in val_stats]).sum()
    tn = torch.cat([x[3] for x in val_stats]).sum()

    val_f1 = smp.metrics.f1_score(tp, fp, fn, tn, reduction="micro")


    scheduler.step(val_f1)

    print(f">> Epoch {epoch+1} Summary | Train Loss: {train_loss/len(train_loader):.4f} | Val F1 Score: {val_f1.item():.4f}")

    # Save Checkpoint
    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save(model.state_dict(), 'best_slum_model.pth')
        print(f"*** New Best Model Saved (F1: {best_f1:.4f}) ***")

Epoch 1/30 [Val]: 100%|██████████| 29/29 [00:05<00:00,  5.60it/s]


>> Epoch 1 Summary | Train Loss: 0.7500 | Val F1 Score: 0.9262
*** New Best Model Saved (F1: 0.9262) ***


Epoch 2/30 [Val]: 100%|██████████| 29/29 [00:06<00:00,  4.33it/s]


>> Epoch 2 Summary | Train Loss: 0.7419 | Val F1 Score: 0.9291
*** New Best Model Saved (F1: 0.9291) ***


Epoch 3/30 [Val]: 100%|██████████| 29/29 [00:05<00:00,  5.39it/s]


...

>> Epoch 28 Summary | Train Loss: 0.7199 | Val F1 Score: 0.9440


Epoch 29/30 [Val]: 100%|██████████| 29/29 [00:05<00:00,  5.63it/s]


>> Epoch 29 Summary | Train Loss: 0.7225 | Val F1 Score: 0.9468


Epoch 30/30 [Val]: 100%|██████████| 29/29 [00:06<00:00,  4.52it/s]


>> Epoch 30 Summary | Train Loss: 0.7189 | Val F1 Score: 0.9482
*** New Best Model Saved (F1: 0.9482) ***


In [ ]:
import matplotlib.pyplot as plt
import random

def visualize_results(model, test_loader, device, num_samples=155):
    model.eval()
    samples_shown = 0

    # Get a random batch from test loader
    with torch.no_grad():
        for imgs, msks in test_loader:
            imgs, msks = imgs.to(device), msks.to(device)
            preds = model(imgs)

            # Move to CPU for plotting
            imgs = imgs.cpu()
            msks = msks.cpu()
            preds = preds.cpu()

            for i in range(imgs.shape[0]):
                if samples_shown >= num_samples:
                    return

                plt.figure(figsize=(15, 5))


                # Undo ImageNet Normalization for display
                img_display = imgs[i].permute(1, 2, 0).numpy()
                img_display = img_display * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
                img_display = np.clip(img_display, 0, 1)

                plt.subplot(1, 3, 1)
                plt.imshow(img_display)
                plt.title("Original Satellite Image")
                plt.axis('off')

                # Plot 2: Ground Truth
                plt.subplot(1, 3, 2)
                plt.imshow(msks[i], cmap='Reds')
                plt.title("Ground Truth (Label)")
                plt.axis('off')


                #  threshold at 0.5 to see the binary mask
                pred_mask = (preds[i][0] > 0.5).float()
                plt.subplot(1, 3, 3)
                plt.imshow(pred_mask, cmap='Blues')
                plt.title(f"Model Prediction (F1: {best_f1:.4f})")
                plt.axis('off')

                plt.show()
                samples_shown += 1


visualize_results(model, test_loader, device)

In [ ]:
import rasterio


image_path = "/content/000002313.tif"

with rasterio.open(image_path) as src:
    print(f"Image path: {image_path}")
    print(f"Image width: {src.width} pixels")
    print(f"Image height: {src.height} pixels")
    print(f"Number of bands (channels): {src.count}")
    print(f"Image CRS (Coordinate Reference System): {src.crs}")
    print(f"Image transform (resolution and origin):\n{src.transform}")


    x_res = src.transform.a
    y_res = abs(src.transform.e)
    print(f"Approximate X-resolution: {x_res} units per pixel")
    print(f"Approximate Y-resolution: {y_res} units per pixel")

Image path: /content/000002313.tif
Image width: 400 pixels
Image height: 400 pixels
Number of bands (channels): 3
Image CRS (Coordinate Reference System): EPSG:32643
Image transform (resolution and origin):
| 0.80, 0.00, 275121.50|
| 0.00,-0.80, 2104045.75|
| 0.00, 0.00, 1.00|
Approximate X-resolution: 0.8000000000000024 units per pixel
Approximate Y-resolution: 0.7999999999999654 units per pixel


In [ ]:
import rasterio


image_path = "/content/tile_5.36.tif"

with rasterio.open(image_path) as src:
    print(f"Image path: {image_path}")
    print(f"Image width: {src.width} pixels")
    print(f"Image height: {src.height} pixels")
    print(f"Number of bands (channels): {src.count}")
    print(f"Image CRS (Coordinate Reference System): {src.crs}")
    print(f"Image transform (resolution and origin):\n{src.transform}")

    x_res = src.transform.a
    y_res = abs(src.transform.e)
    print(f"Approximate X-resolution: {x_res} units per pixel")
    print(f"Approximate Y-resolution: {y_res} units per pixel")

Image path: /content/tile_5.36.tif
Image width: 600 pixels
Image height: 600 pixels
Number of bands (channels): 4
Image CRS (Coordinate Reference System): EPSG:32643
Image transform (resolution and origin):
| 0.57, 0.00, 274388.13|
| 0.00,-0.57, 2106495.24|
| 0.00, 0.00, 1.00|
Approximate X-resolution: 0.5651820000000007 units per pixel
Approximate Y-resolution: 0.5651820000000007 units per pixel


In [ ]:
import os
import shutil
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import rasterio
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A


WEIGHTS_PATH = "/content/best_slum_model.pth"
IMAGE_DIR = "/content/drive/MyDrive/input images"
THRESHOLD = 0.45

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = smp.FPN(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

print("Model loaded")

# -------- PREPROCESS --------
def load_image(tif_path):
    with rasterio.open(tif_path) as src:
        img = src.read()[:3]          # take first 3 channels
        img = img.transpose(1, 2, 0)  # HWC

        img = img / (img.max() + 1e-8)  # normalize
        img = img.astype(np.float32)

    img = A.Resize(256, 256)(image=img)["image"]
    img = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

    return img, img.squeeze().permute(1, 2, 0).cpu().numpy()


tif_files = glob.glob(os.path.join(IMAGE_DIR, "*.tif"))

for tif in tif_files:
    img_tensor, img_vis = load_image(tif)

    with torch.no_grad():
        pred = model(img_tensor)

    mask = (pred.squeeze().cpu().numpy() > THRESHOLD).astype(np.uint8)


    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_vis)
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Predicted Slum Mask")
    plt.axis("off")

    plt.show()


In [ ]:
pip install segmentation_models_pytorch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.8/154.8 kB 4.1 MB/s eta 0:00:00


In [ ]:
import torch
import rasterio
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import segmentation_models_pytorch as smp
import albumentations as A

WEIGHTS_PATH = "/content/best_slum_model.pth"
IMAGE_DIR = "/content/drive/MyDrive/input images"
THRESHOLD = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


model = smp.FPN(
    encoder_name="vgg16",
    encoder_weights="imagenet",
    in_channels=3,
    classes=1,
    activation="sigmoid"
).to(device)

model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=device))
model.eval()

print("Model loaded")

# -------- PREPROCESS --------
def load_image(tif_path):
    with rasterio.open(tif_path) as src:
        img = src.read()[:3]          # take first 3 channels
        img = img.transpose(1, 2, 0)  # HWC

        img = img / (img.max() + 1e-8)  # normalize
        img = img.astype(np.float32)

    img = A.Resize(256, 256)(image=img)["image"]
    img = torch.from_numpy(img).permute(2, 0, 1).unsqueeze(0).to(device)

    return img, img.squeeze().permute(1, 2, 0).cpu().numpy()

# -------- RUN INFERENCE --------
tif_files = glob.glob(os.path.join(IMAGE_DIR, "*.tif"))

for tif in tif_files:
    img_tensor, img_vis = load_image(tif)

    with torch.no_grad():
        pred = model(img_tensor)

    mask = (pred.squeeze().cpu().numpy() > THRESHOLD).astype(np.uint8)

    # -------- SHOW RESULT --------
    plt.figure(figsize=(10, 4))

    plt.subplot(1, 2, 1)
    plt.imshow(img_vis)
    plt.title("Input Image")
    plt.axis("off")

    plt.subplot(1, 2, 2)
    plt.imshow(mask, cmap="gray")
    plt.title("Predicted Slum Mask")
    plt.axis("off")

    plt.show()
